# Full-Scale Data Warehouse + Market Segmentation (K-Means)

All 25 quarterly files (2020 Q1 to 2026 Q1) are loaded with Spark into the PostgreSQL star schema.
Residential districts are then clustered into market segments with Spark MLlib K-Means, and the
segments are written back into the warehouse as the `district_segment` table.

**Before running:** recreate the tables with `sql/create_schema.sql` so this load starts from empty tables.

## 1. Spark session and connection

In [ ]:
import os, sys, glob, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from getpass import getpass
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .appName("DW_full_bonus")
         .master("local[*]")
         .config("spark.driver.extraClassPath", os.path.abspath("../jars/postgresql-42.7.4.jar"))
         .config("spark.driver.memory", "4g")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .config("spark.sql.shuffle.partitions", "32")
         .getOrCreate())

pg_url = os.environ.get("PG_URL", "jdbc:postgresql://localhost:5432/DW")
pg_props = {"user": os.environ.get("PG_USER", "postgres"),
            "password": os.environ.get("PG_PASSWORD") or getpass("Postgres password: "),
            "driver": "org.postgresql.Driver",
            "reWriteBatchedInserts": "true"}
t0 = time.time()

: 

## 2. Extract: detect source formats
The 25 files come in 3 different layouts. Each file is grouped by its number of columns, and each layout is mapped onto one common staging structure.

In [ ]:
files = sorted(glob.glob("../data/*.csv"))
groups = {}
for f in files:
    with open(f, encoding="utf-8-sig") as fh:
        groups.setdefault(len(fh.readline().strip().split(",")), []).append(f)

layouts = {
    10: ["region", "city", "city_district", "deal_id", "date_greg", "date_hijri",
         "property_class", "num_properties", "price", "area"],
    11: ["region", "city", "city_district", "deal_id", "date_greg", "date_hijri",
         "property_class", "property_type", "num_properties", "price", "area"],
    13: ["deal_id", "region", "city", "city_district", "plan", "plot", "date_greg",
         "property_class", "property_type", "num_properties", "price", "area", "price_per_m2_src"],
}
common = ["region", "city", "city_district", "deal_id", "date_greg", "date_hijri",
          "property_class", "num_properties", "price", "area", "source_file"]

for n, paths in groups.items():
    print(f"{n} columns -> {len(paths)} files: {[os.path.basename(p) for p in paths]}")

In [ ]:
def read_layout(paths, names):
    df = (spark.read.csv(paths, header=True, encoding="UTF-8", quote='"', inferSchema=False)
          .toDF(*names)
          .withColumn("source_file", F.col("_metadata.file_name")))
    for c in common:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast("string"))
    return df.select(common)

staging = None
for n, paths in groups.items():
    part = read_layout(paths, layouts[n])
    staging = part if staging is None else staging.unionByName(part)

staging = staging.withColumn("load_ts", F.current_timestamp()).cache()
print("staged rows:", staging.count())

In [ ]:
staging.write.jdbc(pg_url, "stg_deals", mode="overwrite", properties=pg_props)
staging.groupBy("source_file").count().orderBy("source_file").show(30, truncate=False)

## 3. Transform: clean, standardize, validate

In [ ]:
def to_num(c):
    return F.regexp_replace(F.col(c), r"[,\s]", "").cast("decimal(15,2)")

def norm_name(c):
    x = F.trim(F.regexp_replace(F.col(c), r"\s+", " "))
    return F.regexp_replace(x, "ه$", "ة")

clean = (staging
    .withColumn("deal_date",
        F.when(F.col("date_greg").rlike(r"^\d{4}/"), F.to_date("date_greg", "yyyy/M/d"))
         .otherwise(F.to_date("date_greg", "M/d/yyyy")))
    .withColumn("price", to_num("price"))
    .withColumn("area", to_num("area"))
    .withColumn("num_properties", F.col("num_properties").cast("double").cast("int"))
    .withColumn("deal_id", F.regexp_replace("deal_id", r"\.0$", ""))
    .withColumn("deal_id", F.regexp_replace("deal_id", r"\D", "").cast("long"))
    .withColumn("region", norm_name("region"))
    .withColumn("region", F.regexp_replace("region", "^المنطقة ", "منطقة "))
    .withColumn("city", norm_name("city"))
    .withColumn("district", F.trim(F.regexp_extract("city_district", r"/(.*)$", 1)))
    .withColumn("district", F.regexp_replace("district", r"^حي\s+", ""))
    .withColumn("district", F.when(F.col("district") == "", "غير محدد").otherwise(F.col("district")))
    .withColumn("property_class", F.coalesce(F.trim("property_class"), F.lit("غير محدد")))
    .withColumn("price_per_m2", F.round(F.col("price") / F.col("area"), 2))
    .withColumn("file_year", F.regexp_extract("source_file", r"_(\d{4})_q", 1).cast("int"))
    .withColumn("file_quarter", F.regexp_extract("source_file", r"_q(\d)", 1).cast("int"))
)

In [ ]:
business_cols = ["region", "city", "district", "deal_id", "deal_date", "property_class",
                 "num_properties", "price", "area"]

total = clean.count()
clean.select(
    F.lit(total).alias("total_rows"),
    F.sum(F.col("deal_date").isNull().cast("int")).alias("bad_date"),
    F.sum(F.col("price").isNull().cast("int")).alias("bad_price"),
    F.sum(F.col("area").isNull().cast("int")).alias("bad_area"),
    F.sum((F.col("price") <= 0).cast("int")).alias("price_le_0"),
    F.sum((F.col("area") <= 0).cast("int")).alias("area_le_0"),
    F.sum(((F.year("deal_date") != F.col("file_year")) |
           (F.quarter("deal_date") != F.col("file_quarter"))).cast("int")).alias("date_outside_file_quarter"),
).show()

checked = clean.filter(F.col("deal_date").isNotNull() & (F.col("price") > 0) & (F.col("area") > 0))
n_checked = checked.count()
valid = checked.dropDuplicates(business_cols).cache()
n_valid = valid.count()
print("rows failing validation:", total - n_checked)
print("duplicate deals removed:", n_checked - n_valid)
print("valid rows:", n_valid, "| rejected total:", total - n_valid)

## 4. Build dimensions and fact

In [ ]:
dim_date = (valid.select(F.col("deal_date").alias("full_date")).distinct()
    .select(F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
            "full_date",
            F.dayofmonth("full_date").alias("day"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.quarter("full_date").alias("quarter"),
            F.year("full_date").alias("year")))

dim_location = (valid.select("region", "city", "district").distinct()
    .withColumn("location_key", F.row_number().over(Window.orderBy("region", "city", "district")))
    .select("location_key", "region", "city", "district")
    .cache())

dim_property = (valid.select("property_class").distinct()
    .withColumn("property_key", F.row_number().over(Window.orderBy("property_class")))
    .select("property_key", "property_class"))

dim_source = (valid.select("source_file", "file_year", "file_quarter").distinct()
    .withColumn("source_key", F.row_number().over(Window.orderBy("source_file")))
    .select("source_key", "source_file",
            F.col("file_year").alias("year"), F.col("file_quarter").alias("quarter")))

fact = (valid
    .join(dim_location, ["region", "city", "district"])
    .join(dim_property, ["property_class"])
    .join(dim_source.select("source_key", "source_file"), ["source_file"])
    .withColumn("date_key", F.date_format("deal_date", "yyyyMMdd").cast("int"))
    .withColumn("deal_key", F.monotonically_increasing_id() + 1)
    .select("deal_key", "deal_id", "date_key", "location_key", "property_key", "source_key",
            "num_properties", "price", "area", "price_per_m2")
    .cache())

for name, d in [("dim_date", dim_date), ("dim_location", dim_location), ("dim_property", dim_property),
                ("dim_source", dim_source), ("fact_deals", fact)]:
    print(f"{name:13s} {d.count():>10,}")

## 5. Load into PostgreSQL (dimensions first, then fact)

In [ ]:
for name, d in [("dim_date", dim_date), ("dim_location", dim_location), ("dim_property", dim_property),
                ("dim_source", dim_source), ("fact_deals", fact)]:
    d.write.jdbc(pg_url, name, mode="append", properties=pg_props)
    print("loaded", name)
print(f"ETL time: {time.time() - t0:.0f} s")

## 6. Data mining: K-Means market segmentation of districts
Each residential district becomes one data point described by 4 features:

| Feature | Meaning |
|---|---|
| median price per m² | how expensive the district is |
| number of deals | how active the market is |
| median plot area | typical property size |
| price growth | median price per m² in 2024-2026 divided by 2020-2021 |

Log transform reduces skew, StandardScaler puts all features on the same scale, and k is chosen with the elbow method and silhouette score: the smallest k between 3 and 6 whose silhouette is within 0.01 of the best, so segments stay easy to interpret.

In [ ]:
yr = F.col("year")
res = (fact.join(dim_property, "property_key")
           .join(dim_date.select("date_key", "year"), "date_key")
           .join(dim_location, "location_key")
           .filter((F.col("property_class") == "سكني") &
                   (F.col("district") != "غير محدد") &
                   F.col("price_per_m2").between(10, 50000)))

district_features = (res.groupBy("location_key")
    .agg(F.count("*").alias("deals"),
         F.percentile_approx("price_per_m2", 0.5).alias("median_ppm"),
         F.percentile_approx("area", 0.5).alias("median_area"),
         F.percentile_approx(F.when(yr <= 2021, F.col("price_per_m2")), 0.5).alias("ppm_early"),
         F.percentile_approx(F.when(yr >= 2024, F.col("price_per_m2")), 0.5).alias("ppm_recent"),
         F.sum((yr <= 2021).cast("int")).alias("n_early"),
         F.sum((yr >= 2024).cast("int")).alias("n_recent"))
    .filter((F.col("deals") >= 30) & (F.col("n_early") >= 5) & (F.col("n_recent") >= 5))
    .withColumn("price_growth", F.round(F.col("ppm_recent") / F.col("ppm_early"), 3))
    .withColumn("f_price", F.log("median_ppm"))
    .withColumn("f_activity", F.log("deals"))
    .withColumn("f_area", F.log("median_area"))
    .withColumn("f_growth", F.log(F.least(F.greatest("price_growth", F.lit(0.2)), F.lit(5.0))))
    .cache())
print("districts used for clustering:", district_features.count())

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

assembler = VectorAssembler(inputCols=["f_price", "f_activity", "f_area", "f_growth"], outputCol="raw")
scaler = StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True)
assembled = assembler.transform(district_features)
X = scaler.fit(assembled).transform(assembled).cache()

evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette")
scores = []
for k in range(2, 9):
    m = KMeans(k=k, seed=42, featuresCol="features").fit(X)
    scores.append((k, m.summary.trainingCost, evaluator.evaluate(m.transform(X))))
    print(f"k={k}  inertia={scores[-1][1]:9.1f}  silhouette={scores[-1][2]:.3f}")

candidates = [s for s in scores if 3 <= s[0] <= 6]
top = max(c[2] for c in candidates)
best_k = min(c[0] for c in candidates if c[2] >= top - 0.01)
print("chosen k:", best_k, "(smallest k in 3-6 within 0.01 of the best silhouette)")

In [ ]:
import matplotlib.pyplot as plt
ks, inertia, sil = zip(*scores)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ks, inertia, marker="o"); ax[0].set_title("Elbow method"); ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia (within-cluster SSE)")
ax[1].plot(ks, sil, marker="o", color="tab:green"); ax[1].set_title("Silhouette score"); ax[1].set_xlabel("k")
ax[1].axvline(best_k, ls="--", color="gray")
plt.tight_layout(); plt.savefig("../docs/images/kmeans_k_selection.png", dpi=150); plt.show()

In [ ]:
model = KMeans(k=best_k, seed=42, featuresCol="features").fit(X)
clustered = model.transform(X).withColumnRenamed("prediction", "cluster")

profile = (clustered.groupBy("cluster")
    .agg(F.count("*").alias("districts"),
         F.round(F.expr("percentile_approx(median_ppm, 0.5)"), 0).alias("median_ppm"),
         F.round(F.expr("percentile_approx(deals, 0.5)"), 0).alias("median_deals"),
         F.round(F.expr("percentile_approx(median_area, 0.5)"), 0).alias("median_area"),
         F.round(F.expr("percentile_approx(price_growth, 0.5)"), 2).alias("median_growth"))
    .toPandas().sort_values("median_ppm").reset_index(drop=True))

def name_segments(p):
    names = {p.index[0]: "Affordable", p.index[-1]: "Premium"}
    middle = p.iloc[1:-1]
    if len(middle):
        busiest = middle["median_deals"].idxmax()
        names[busiest] = "High-demand"
        rest = middle.drop(busiest)
        if len(rest):
            names[rest["median_growth"].idxmax()] = "Emerging (fast growth)"
        for i, idx in enumerate(i for i in middle.index if i not in names):
            names[idx] = f"Mid-market {i + 1}"
    return [names[i] for i in p.index]

profile["segment_id"] = range(1, best_k + 1)
profile["segment_name"] = name_segments(profile)
segment_order = list(profile["segment_name"])
profile

In [ ]:
mapping = spark.createDataFrame(profile[["cluster", "segment_id", "segment_name"]].astype({"cluster": int, "segment_id": int}))
district_segment = (clustered.join(mapping, "cluster")
    .select("location_key", "segment_id", "segment_name", "deals",
            F.col("median_ppm").cast("decimal(14,2)").alias("median_price_per_m2"),
            F.col("median_area").cast("decimal(14,2)").alias("median_area"),
            F.col("price_growth").cast("decimal(8,3)")))
district_segment.write.jdbc(pg_url, "district_segment", mode="append", properties=pg_props)
print("loaded district_segment:", district_segment.count())

In [ ]:
pdf = clustered.join(mapping, "cluster").select("median_ppm", "price_growth", "deals", "segment_name").toPandas()
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name in segment_order:
    s = pdf[pdf.segment_name == name]
    ax[0].scatter(s.median_ppm, s.price_growth.clip(0.2, 6), s=10, alpha=0.6, label=name)
    ax[1].scatter(s.median_ppm, s.deals, s=10, alpha=0.6, label=name)
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("Median price per m² (SAR)"); ax[0].set_ylabel("Price growth 2024-26 vs 2020-21")
ax[0].axhline(1, ls="--", color="gray", lw=0.8); ax[0].set_title("Price level vs growth")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("Median price per m² (SAR)"); ax[1].set_ylabel("Number of deals 2020-2026")
ax[1].set_title("Price level vs market activity"); ax[1].legend()
fig.suptitle("Residential district market segments (K-Means, k=%d)" % best_k)
plt.tight_layout(); plt.savefig("../docs/images/kmeans_segments.png", dpi=150); plt.show()

In [ ]:
(district_segment.join(dim_location, "location_key")
    .filter(F.col("city") == "الرياض")
    .withColumn("rn", F.row_number().over(Window.partitionBy("segment_name").orderBy(F.desc("deals"))))
    .filter("rn <= 3")
    .select("segment_name", "district", "deals", "median_price_per_m2", "price_growth")
    .orderBy("median_price_per_m2")
    .show(30, truncate=False))
print(f"Total time: {time.time() - t0:.0f} s")